# Phân tích chẩn đoán

## 1. Chuẩn bị dữ liệu

In [1]:
# Load libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [2]:
customers = pd.read_csv("../data/2_clean/customers.csv")
orders = pd.read_csv("../data/2_clean/orders.csv")
order_items = pd.read_csv("../data/2_clean/order_items.csv")
payments = pd.read_csv("../data/2_clean/payments.csv")
products = pd.read_csv("../data/2_clean/products.csv")
reviews = pd.read_csv("../data/2_clean/reviews.csv")
geolocation = pd.read_csv("../data/2_clean/geolocation.csv")
sellers = pd.read_csv("../data/2_clean/sellers.csv")

## 2. Phân loại cảm xúc
**Mục tiêu:** phân loại các đánh giá dưới 3 sao theo `negative / positive / neutral`

In [3]:
reviews

,review_id,order_id,review_score,review_comment_message
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,Recebi bem antes do prazo estipulado.
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,Parabéns lojas lannister adorei comprar pela I...
...,...,...,...,...
99219,574ed12dd733e5fa530cfd4bbf39d7c9,2a8c23fee101d4d5662fa670396eb8da,5,NaN
99220,f3897127253a9592a73be9bdfdf4ed7a,22ec9f0669f784db00fa86d035cf8602,5,NaN
99221,b3de70c89b1510c4cd3d0649fd302472,55d4004744368f5571d1f590031933e4,5,"Excelente mochila, entrega super rápida. Super..."
99222,1adeb9d84d72fe4e337617733eb85149,7725825d039fc1f0ceb7635e3f7d9206,4,NaN


In [4]:
df = reviews[(~reviews["review_comment_message"].isnull()) & (reviews["review_score"] < 3)]
display(df[["review_id", "review_comment_message"]])

,review_id,review_comment_message
16,9314d6f9799f5bfba510cc7bcd468c01,"GOSTARIA DE SABER O QUE HOUVE, SEMPRE RECEBI E..."
19,373cbeecea8286a2b66c97b1b157ec46,Péssimo
29,2c5e27fc178bde7ac173c9c62c31b070,Não gostei ! Comprei gato por lebre
32,58044bca115705a48fe0e00a21390c54,Sempre compro pela Internet e a entrega ocorre...
39,9fd59cd04b42f600df9f25e54082a8d1,Nada de chegar o meu pedido.
...,...,...
99155,2fc8295a24978092328d3a329d22940e,Eu recebi o seguinte email e preciso saber com...
99169,3450ec3bbabeb09a08b00fc72da87a30,Boa tarde. \r\nNão recebo todos os produtos fa...
99200,2ee221b28e5b6fceffac59487ed39348,Foto muito diferente principalmente a graninha...
99203,5085bc489aa6b58a29c4f922d59ff826,Tive um problema na entrega em que o correio c...


In [5]:
#!pip install pysentimiento   

In [6]:
#!pip install --upgrade accelerate

In [7]:
#!pip install --upgrade torch transformers

In [8]:
import accelerate
import torch
import transformers

print(accelerate.__version__)    # >=0.26.0
print(torch.__version__)         # >=2.0
print(transformers.__version__)  # >=4.13

c:\Users\sonvuuu\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1.12.0
2.9.1+cpu
4.57.3


In [7]:
import pandas as pd
from pysentimiento import create_analyzer

# Tạo analyzer sentiment cho tiếng Bồ Đào Nha
analyzer = create_analyzer(task="sentiment", lang="pt")

# Hàm map label sang Positive / Neutral / Negative
def map_label(label):
    return {
        "NEG": "Negative",
        "NEU": "Neutral",
        "POS": "Positive"
    }.get(label, "Neutral")

# Batch size
batch_size = 32
sentiments = []

# Chia comment thành batch
comments = df['review_comment_message'].tolist()
for i in range(0, len(comments), batch_size):
    batch = comments[i:i+batch_size]
    results = analyzer.predict(batch)  # pysentimiento hỗ trợ batch
    sentiments.extend([map_label(r.output) for r in results])

# Gán kết quả vào DataFrame
df['sentiment'] = sentiments

# Hiển thị 5 dòng đầu
print(df.head())

c:\Users\sonvuuu\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Map: 100%|██████████| 32/32 [00:00<00:00, 2279.24 examples/s]
c:\Users\sonvuuu\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Map: 100%|██████████| 10/10 [00:00<00:00, 1514.30 examples/s]


                           review_id                          order_id  \
16  9314d6f9799f5bfba510cc7bcd468c01  0dacf04c5ad59fd5a0cc1faa07c34e39   
19  373cbeecea8286a2b66c97b1b157ec46  583174fbe37d3d5f0d6661be3aad1786   
29  2c5e27fc178bde7ac173c9c62c31b070  0ce9a24111d850192a933fcaab6fbad3   
32  58044bca115705a48fe0e00a21390c54  68e55ca79d04a79f20d4bfc0146f4b66   
39  9fd59cd04b42f600df9f25e54082a8d1  3c314f50bc654f3c4e317b055681dff9   

    review_score                             review_comment_message sentiment  
16             2  GOSTARIA DE SABER O QUE HOUVE, SEMPRE RECEBI E...  Negative  
19             1                                            Péssimo  Negative  
29             1                Não gostei ! Comprei gato por lebre  Negative  
32             1  Sempre compro pela Internet e a entrega ocorre...  Negative  
39             1                       Nada de chegar o meu pedido.  Negative  


C:\Users\sonvuuu\AppData\Local\Temp\ipykernel_10028\2771575334.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['sentiment'] = sentiments


In [8]:
df

,review_id,order_id,review_score,review_comment_message,sentiment
16,9314d6f9799f5bfba510cc7bcd468c01,0dacf04c5ad59fd5a0cc1faa07c34e39,2,"GOSTARIA DE SABER O QUE HOUVE, SEMPRE RECEBI E...",Negative
19,373cbeecea8286a2b66c97b1b157ec46,583174fbe37d3d5f0d6661be3aad1786,1,Péssimo,Negative
29,2c5e27fc178bde7ac173c9c62c31b070,0ce9a24111d850192a933fcaab6fbad3,1,Não gostei ! Comprei gato por lebre,Negative
32,58044bca115705a48fe0e00a21390c54,68e55ca79d04a79f20d4bfc0146f4b66,1,Sempre compro pela Internet e a entrega ocorre...,Negative
39,9fd59cd04b42f600df9f25e54082a8d1,3c314f50bc654f3c4e317b055681dff9,1,Nada de chegar o meu pedido.,Negative
...,...,...,...,...,...
99155,2fc8295a24978092328d3a329d22940e,e809c167a9cfd31aee1293abe8995065,1,Eu recebi o seguinte email e preciso saber com...,Neutral
99169,3450ec3bbabeb09a08b00fc72da87a30,c46f950813dd2dd9bab4188dceb83175,1,Boa tarde. \r\nNão recebo todos os produtos fa...,Neutral
99200,2ee221b28e5b6fceffac59487ed39348,f2d12dd37eaef72ed7b1186b2edefbcd,2,Foto muito diferente principalmente a graninha...,Neutral
99203,5085bc489aa6b58a29c4f922d59ff826,18ed848509774f56cc8c1c0a1903ad7f,2,Tive um problema na entrega em que o correio c...,Negative


In [9]:
import os

# Lưu từng DataFrame
output_dir = "../data/3_sentiment"

df.to_csv(os.path.join(output_dir, "reviews_1.csv"), index=False)

In [43]:
df = pd.read_csv("../data/3_sentiment/reviews_1.csv")
df = df[df["sentiment"] == "Positive"]
df

,review_id,order_id,review_score,review_comment_message,sentiment
80,7b66de927426b71a817aa36df5e8a0b3,634e8f4c0f6744a626f77f39770ac6aa,1,Muito bom o produto e preço cobrado!,Positive
85,8bc4d875dbab927e9190060688edc80f,b6e5ddeaf2e5b56015e295beb333f305,2,Excelente produto indico a qualquer pessoa.,Positive
126,b42955e5003fa4e9449770fb8b50d30d,1f186ad4ad389666cd9482ce1942038d,2,Demorou mas entregou!,Positive
258,d04bfe3b90e5436b03b65e86d8b6e62d,9292f2ff747a4f8c6580fc8377bb1c8b,2,Apesar da demora na entrega. No mais foi tudo ...,Positive
314,488f5ad66f112743c91ae61731522375,5e3ec924bc1a1f3793e12632c2b2f85d,1,muito obrigado\r\n,Positive
...,...,...,...,...,...
10532,14876c10d791683465f83751ff33064d,296752c5e35a99dd06b9100ab400c8eb,2,Produto bom,Positive
10540,0ba0eb2b255cfe3925088677a49add35,cf5439c6b13deafe107ecca7dcc58b80,1,"paguei 80 rreais de frete, quero esse produro ...",Positive
10626,c0881b2797529c9af9325018e7fbb694,fd7eb2117233aa5ac29e94a28e082077,1,o produto era exatamente o que eu precisasa,Positive
10636,9740c39fe70be3fdd5b610c34358a278,be73d4698f2f4939662dd7788cf3dad3,2,Até as que dão no lava car e postos de gasolin...,Positive


## 3. Phân loại thể loại
**Mục tiêu:** phân loại các đánh giá dưới 3 sao theo `negative / positive / neutral`

In [59]:
df = pd.read_csv("../data/3_sentiment/reviews_1.csv")
df = df[df["sentiment"] != "Positive"]
df = df.head(20)
df

,review_id,order_id,review_score,review_comment_message,sentiment
0,9314d6f9799f5bfba510cc7bcd468c01,0dacf04c5ad59fd5a0cc1faa07c34e39,2,"GOSTARIA DE SABER O QUE HOUVE, SEMPRE RECEBI E...",Negative
1,373cbeecea8286a2b66c97b1b157ec46,583174fbe37d3d5f0d6661be3aad1786,1,Péssimo,Negative
2,2c5e27fc178bde7ac173c9c62c31b070,0ce9a24111d850192a933fcaab6fbad3,1,Não gostei ! Comprei gato por lebre,Negative
3,58044bca115705a48fe0e00a21390c54,68e55ca79d04a79f20d4bfc0146f4b66,1,Sempre compro pela Internet e a entrega ocorre...,Negative
4,9fd59cd04b42f600df9f25e54082a8d1,3c314f50bc654f3c4e317b055681dff9,1,Nada de chegar o meu pedido.,Negative
5,e233e51d11511bf30e568c76360ace52,548df2c6e5f089574614894bca78acf5,1,recebi somente 1 controle Midea Split ESTILO.\...,Negative
6,6d06808638ec0701bccd70bc8d462c28,97d2f8fe76f2f253b8291e17b5383884,1,O produto não chegou no prazo estipulado e cau...,Negative
7,60c714ed14cef913944a3147094a4742,9ac05114800f02bfaa783bd76842dbe2,1,"Produto muito inferior, mal acabado.",Negative
8,65dfeb60c40e3cbb0a1838285d86f885,a2714ecbf6eeb3bb9cd7dba6dc1c5e82,1,Pedi reembolso e sem resposta até momento,Negative
9,ae728c1061bf163b4bd256ad9ee0bb83,45c780334bc32cb77559a65c5f171160,1,Este foi o pedido\r\nBalde Com 128 Peças - Blo...,Neutral


In [48]:
import requests
import json

system_prompt = """
You are a strict classification engine.

Your task:
- Classify e-commerce user comments into EXACTLY ONE category.
- Focus ONLY on the main complaint or feedback.
- Ignore greetings, politeness, or irrelevant details.

Categories (choose ONE only):
1. Product Quality
   - Defects, broken items, poor quality, wrong product, missing parts

2. Delivery
   - Late delivery, shipping delay, lost package, damaged in transport

3. Customer Service
   - Rude staff, no response, bad support, cannot contact seller/platform

4. Other
   - Praise, neutral comments, unclear feedback, or anything not above

Rules:
- Always output EXACTLY one category name.
- Do NOT explain.
- Do NOT add punctuation.
- Do NOT output numbering.
- Do NOT output multiple labels.
- If unsure, choose "Other".
"""


def translate_comment():
    url = "http://172.30.224.1:1234/v1/responses"
    headers = {"Content-Type": "application/json"}  # nếu có token thì thêm Authorization
    payload = {
        "model": "llama-3-groq-8b-tool-use:2",  
        "input": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": 
                """Produto muito inferior, mal acabado."""}
        ]
    }

    response = requests.post(url, headers=headers, json=payload)

    # Format output
    data = response.json()["output"][0]["content"][0]["text"]

    return data

print(translate_comment())

Product Quality


In [60]:
import pandas as pd
import requests
import time

# ================== CONFIG ==================

URL = "http://172.30.224.1:1234/v1/responses"
MODEL = "llama-3-groq-8b-tool-use:2"

system_prompt = """
You are a strict classification engine.

Your task:
- Classify e-commerce user comments into EXACTLY ONE category.
- Focus ONLY on the main complaint or feedback.
- Ignore greetings, politeness, or irrelevant details.

Categories:
Product Quality
Delivery
Customer Service
Other

Rules:
- Output EXACTLY one category name
- Do NOT explain
- Do NOT add punctuation
- Do NOT add numbering
- Do NOT output multiple labels
- If unsure, choose Other
"""

# ================== CLASSIFY FUNCTION ==================

def classify_comment(text: str) -> str:
    payload = {
        "model": MODEL,
        "input": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": text}
        ]
    }

    try:
        res = requests.post(URL, json=payload, timeout=30)
        res.raise_for_status()
        label = res.json()["output"][0]["content"][0]["text"].strip()

        # Safety clamp
        if label not in ["Product Quality", "Delivery", "Customer Service", "Other"]:
            return "Other"

        return label

    except Exception as e:
        print("Error:", e)
        return "Other"


# ================== RUN (SAFE MODE) ==================

df["category"] = df["review_comment_message"].apply(classify_comment)

df


,review_id,order_id,review_score,review_comment_message,sentiment,category
0,9314d6f9799f5bfba510cc7bcd468c01,0dacf04c5ad59fd5a0cc1faa07c34e39,2,"GOSTARIA DE SABER O QUE HOUVE, SEMPRE RECEBI E...",Negative,Customer Service
1,373cbeecea8286a2b66c97b1b157ec46,583174fbe37d3d5f0d6661be3aad1786,1,Péssimo,Negative,Delivery
2,2c5e27fc178bde7ac173c9c62c31b070,0ce9a24111d850192a933fcaab6fbad3,1,Não gostei ! Comprei gato por lebre,Negative,Customer Service
3,58044bca115705a48fe0e00a21390c54,68e55ca79d04a79f20d4bfc0146f4b66,1,Sempre compro pela Internet e a entrega ocorre...,Negative,Delivery
4,9fd59cd04b42f600df9f25e54082a8d1,3c314f50bc654f3c4e317b055681dff9,1,Nada de chegar o meu pedido.,Negative,Delivery
5,e233e51d11511bf30e568c76360ace52,548df2c6e5f089574614894bca78acf5,1,recebi somente 1 controle Midea Split ESTILO.\...,Negative,Delivery
6,6d06808638ec0701bccd70bc8d462c28,97d2f8fe76f2f253b8291e17b5383884,1,O produto não chegou no prazo estipulado e cau...,Negative,Delivery
7,60c714ed14cef913944a3147094a4742,9ac05114800f02bfaa783bd76842dbe2,1,"Produto muito inferior, mal acabado.",Negative,Delivery
8,65dfeb60c40e3cbb0a1838285d86f885,a2714ecbf6eeb3bb9cd7dba6dc1c5e82,1,Pedi reembolso e sem resposta até momento,Negative,Delivery
9,ae728c1061bf163b4bd256ad9ee0bb83,45c780334bc32cb77559a65c5f171160,1,Este foi o pedido\r\nBalde Com 128 Peças - Blo...,Neutral,Delivery


### 1. Chẩn đoán "Nội dung đánh giá"
**Mục tiêu**: Giải mã lý do thực sự đằng sau các con số 1 sao, 2 sao. Dữ liệu số (Rating) chỉ cho biết mức độ, dữ liệu chữ (Comment) mới cho biết nguyên nhân.
- Trong số các đánh giá tiêu cực (1-2 sao), bao nhiêu % nhắc đến từ khóa liên quan "logistics" (chậm, chưa nhận được), bao nhiêu % liên quan đến "product" (hỏng, sai màu, hàng giả)?
- Có trường hợp nào khách đánh giá 5 sao nhưng comment phàn nàn (hoặc ngược lại) không? Tại sao?
- Những chủ đề (Topic) chính mà khách hàng thường thảo luận là gì? (Ví dụ: Topic 1: Giao hàng, Topic 2: Chất lượng, Topic 3: Dịch vụ CSKH).